In [1]:
import os
os.chdir("/content/drive/MyDrive/RAG-based-AI-Teaching-Assistant")
print("current dir :-", os.getcwd())

current dir :- /content/drive/MyDrive/RAG-based-AI-Teaching-Assistant


In [ ]:
# download embedding model from hugging face
!pip install -U FlagEmbedding
from FlagEmbedding import BGEM3FlagModel
embedding_model = BGEM3FlagModel('BAAI/bge-m3',
                       use_fp32=True)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [38]:
# take argument as text and return its vector
def create_embedding(text):
  embeddings = embedding_model.encode(text)['dense_vecs']
  return embeddings

In [ ]:
# import libraries
import pandas as pd
import joblib
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [56]:
# load embedding data
df = joblib.load('embedding_data.joblib')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2316 entries, 0 to 2315
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   number     2316 non-null   object 
 1   title      2316 non-null   object 
 2   start      2316 non-null   float64
 3   end        2316 non-null   float64
 4   text       2316 non-null   object 
 5   chunks_id  2316 non-null   int64  
 6   embedding  2316 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 126.8+ KB


In [53]:
def create_prompt():
  incoming_query = input("Ask a question :- ")
  question_embdding = create_embedding([incoming_query])[0]
  similarities = cosine_similarity(np.vstack(df['embedding']),[question_embdding]).flatten()
  top_result = 4 # let 4
  top_three_max_idx = similarities.argsort()[::-1][0:top_result]

  # getting data based on cosine similarity
  new_df = df.loc[top_three_max_idx]
  # print(new_df[["number","title","start","end","text"]])

  prompt = f'''This is Sigma Web Development course by Harrry Bhai. Here are the video subtitle chunks
  containing video title, video number, start and end time in second, the text at that time :

  {new_df[["title", "number", "start", "end", "text"]].to_json(orient="records",)}

  "{incoming_query}" — User asked this question, from this video chunks, you have to answer where and how
  much content is taught in which video (and at what timestamp) and guide the user to go to
  that particular video. If user asks unrelated question, tell him that you can only answer the question related to the course
  '''
  return prompt

# for index, item in new_df.iterrows():
#   print(index, item['title'],item['number'],item['start'],item['end'],item['text'])

In [29]:
# Installing Dependencies
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 59 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (511 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Cleaning up old version at /us

In [31]:
# In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built
# to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service
# using subprocess in Python so it doesn't block any cell from running.
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [33]:
# Download the LLM model
!ollama pull gemma3:4b

In [32]:
!pip install langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-ollama]


In [50]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

In [54]:
prompt = create_prompt()
template = """Question: {question}
Answer: """

prompt_template = ChatPromptTemplate.from_template(template)
model = OllamaLLM(model="gemma3:4b")
chain = prompt_template | model

print(chain.invoke({"question": prompt}))

Ask a question :- Where harry vai tell about form and input tags ?
Okay, let’s find out where Harry Bhai talks about forms and input tags.

Here’s what the subtitles tell us:

*   **Video 5: “forms_and_input_tags”** – This video directly covers “what are forms and what is the input tag in the form.” It lasts from 0.0 seconds to 4.9 seconds.

To start with the basics of forms and input tags, you should watch **Video 5**.

Do you have any other questions related to this course, specifically about the content within these videos?
